# 00 — Environment, protocol status and reproducibility

Run this notebook before a controlled experiment. It records the software environment and checks the locked H1-N preprocessing contract. It contains no model-accuracy claim.

**Status discipline.** D0 denotes the completed legacy diagnostic controls that exposed a geometry/source confound. H1-N denotes the amended, source-normalised comparison. D0 metrics are not H1-N results, and no completed H1-N neural run is assumed by this notebook. A model is not selected for an interface until the separately locked external evaluation is complete.

In [ ]:
from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
import torch

from ai_image_detector.features import (
    CONTROLLED_IMAGE_SIZE,
    CONTROLLED_PREPROCESSING_PROTOCOL,
    preprocessing_metadata,
)
from ai_image_detector.reproducibility import environment_snapshot, get_device, save_json, seed_everything

SEED = 7
seed_everything(SEED)
DEVICE = get_device()
H1N_PREPROCESSING = preprocessing_metadata(CONTROLLED_PREPROCESSING_PROTOCOL)

assert H1N_PREPROCESSING['image_size'] == CONTROLLED_IMAGE_SIZE == 128
assert H1N_PREPROCESSING['train_crop'] == 'seeded_random_square_crop'
assert H1N_PREPROCESSING['eval_crop'] == 'center_square_crop'

snapshot = environment_snapshot() | {
    'seed': SEED,
    'selected_device': str(DEVICE),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'h1n_preprocessing': H1N_PREPROCESSING,
}
save_json(snapshot, Path('../artifacts/environment/environment.json'))
snapshot

In [ ]:
artifact_root = Path('../artifacts')
completed_h1n = []
for run_path in artifact_root.glob('*/run.json'):
    run = json.loads(run_path.read_text(encoding='utf-8'))
    protocol = run.get('preprocessing', {}).get('protocol')
    metrics_path = run_path.parent / 'internal_test_metrics.json'
    if protocol == CONTROLLED_PREPROCESSING_PROTOCOL and metrics_path.is_file():
        completed_h1n.append(run_path.parent.name)

study_status = {
    'D0_legacy_diagnostics': 'completed; retained only as a confound diagnostic',
    'H1_N_completed_internal_runs': sorted(completed_h1n),
    'H1_N_confirmatory_external_evaluation': 'locked and pending',
    'deployable_model': 'none until the frozen external evaluation is reported',
}
study_status

## Acceptance checks

A controlled run may proceed only when the device and Git revision are recorded, the H1-N metadata says `h1n_square_crop_128_v1` and `128 × 128`, and the grouped-manifest gate in Notebook 01 passes. For neural H1-N RGB/FFT training, the train crop is a seeded random square crop; neural validation and evaluation use the deterministic centre crop. The controlled radial-logistic baseline is intentionally different: it extracts deterministic centre-crop features on train, validation, and test, with no augmentation. On Apple Silicon, record the MPS availability; do not treat CPU/MPS choice as a performance result.

In [ ]:
assert torch.__version__, 'PyTorch is unavailable'
print(json.dumps(snapshot, indent=2))
print('MPS available:', torch.backends.mps.is_available())
print('No accuracy, calibration, or deployment claim is produced in this notebook.')